# Spectral - Anomaly Detection Demo

This notebook walks through the core detection pipeline: generating synthetic data, running individual detectors, comparing their performance, and using the ensemble.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from spectral.ingestion.synthetic import SyntheticSource
from spectral.detectors import ZScoreDetector, MADDetector, IsolationForestDetector, SpectralResidualDetector
from spectral.detectors.ensemble import EnsembleDetector
from spectral.config import get_default_config
from spectral.pipeline.runner import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)

## Generate synthetic data

Create a time series with known anomalies so we can evaluate detector performance against ground truth.

In [2]:
source = SyntheticSource(seed=42)
data_points, labels = source.generate(
    n_points=2000,
    anomaly_ratio=0.03,
    noise_level=0.1,
    patterns=['sine', 'seasonal'],
)

values = np.array([p.value for p in data_points])
timestamps = [p.timestamp for p in data_points]
anomaly_mask = labels.astype(bool)

print(f'Generated {len(values)} points with {labels.sum()} anomalies ({labels.mean():.1%})')

Generated 2000 points with 58 anomalies (2.9%)


In [3]:
fig, ax = plt.subplots()
ax.plot(values, linewidth=0.5, alpha=0.7, label='signal')
ax.scatter(np.where(anomaly_mask)[0], values[anomaly_mask], 
           c='red', s=20, zorder=5, label='true anomalies')
ax.set_xlabel('Time step')
ax.set_ylabel('Value')
ax.set_title('Synthetic time series with injected anomalies')
ax.legend()
plt.tight_layout()
plt.show()

<Figure size 1400x400 with 1 Axes>

## Run individual detectors

In [4]:
detectors = [
    ZScoreDetector(threshold=3.0),
    MADDetector(threshold=3.5),
    IsolationForestDetector(contamination=0.05),
    SpectralResidualDetector(window_size=24, score_threshold=3.0),
]

results = {}
for det in detectors:
    result = det.fit_detect(values)
    results[det.name] = result
    print(f'{det.name + ":":26s} {result.anomaly_count} anomalies detected')

ZScoreDetector:           27 anomalies detected
MADDetector:              31 anomalies detected
IsolationForestDetector:  100 anomalies detected
SpectralResidualDetector: 19 anomalies detected


In [5]:
fig, axes = plt.subplots(len(detectors), 1, figsize=(14, 2.5 * len(detectors)), sharex=True)

for ax, (name, result) in zip(axes, results.items()):
    ax.plot(result.scores, linewidth=0.5, alpha=0.7)
    detected = result.anomaly_indices
    ax.scatter(detected, result.scores[detected], c='red', s=15, zorder=5)
    ax.set_ylabel('Score')
    ax.set_title(name)

axes[-1].set_xlabel('Time step')
plt.suptitle('Anomaly scores by detector', y=1.02)
plt.tight_layout()
plt.show()

<Figure size 1400x1000 with 4 Axes>

## Evaluate against ground truth

In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(f'{"Detector":27s} {"Precision":11s} {"Recall":11s} {"F1":10s}')
print('-' * 62)

for name, result in results.items():
    p = precision_score(labels, result.labels, zero_division=0)
    r = recall_score(labels, result.labels, zero_division=0)
    f1 = f1_score(labels, result.labels, zero_division=0)
    print(f'{name:27s} {p:<11.3f} {r:<11.3f} {f1:<10.3f}')

Detector                   Precision  Recall     F1        
--------------------------------------------------------------
ZScoreDetector             0.741      0.345      0.470     
MADDetector                0.710      0.379      0.494     
IsolationForestDetector    0.260      0.448      0.329     
SpectralResidualDetector   0.684      0.224      0.338     


## Ensemble detection

Combine multiple detectors to reduce false positives. With `method='voting'` and `min_detectors=2`, a point is flagged only if at least 2 detectors agree.

In [7]:
ensemble = EnsembleDetector(
    detectors=[
        ZScoreDetector(threshold=3.0),
        MADDetector(threshold=3.5),
        IsolationForestDetector(contamination=0.05),
        SpectralResidualDetector(window_size=24),
    ],
    method='voting',
    min_detectors=2,
)

ensemble_result = ensemble.fit_detect(values)

p = precision_score(labels, ensemble_result.labels, zero_division=0)
r = recall_score(labels, ensemble_result.labels, zero_division=0)
f1 = f1_score(labels, ensemble_result.labels, zero_division=0)

print(f'Ensemble: {ensemble_result.anomaly_count} anomalies detected')
print(f'  Precision: {p:.3f}')
print(f'  Recall:    {r:.3f}')
print(f'  F1:        {f1:.3f}')

Ensemble: 24 anomalies detected
  Precision: 0.833
  Recall:    0.345
  F1:        0.488


The ensemble achieves higher precision than any individual detector by requiring consensus. The trade-off is slightly lower recall, but in production alerting, fewer false positives is usually worth it.

## Full pipeline

The `Pipeline` class wraps everything together: preprocessing, detection, and severity scoring.

In [8]:
config = get_default_config()
pipeline = Pipeline(config)
anomalies = pipeline.detect(values, timestamps)

print(f'Pipeline detected {len(anomalies)} anomalies')

# severity breakdown
from collections import Counter
severity_counts = Counter(a.severity for a in anomalies)
print('\nSeverity breakdown:')
for sev in ['critical', 'high', 'warning', 'low']:
    print(f'  {sev}: {severity_counts.get(sev, 0)}')

print('\nSample anomalies:')
for a in anomalies[:5]:
    print(f'  idx={a.index:<5d} value={a.value:<8.4f} score={a.score:.3f}  severity={a.severity:<10s} detector={a.detector}')

Pipeline detected 24 anomalies

Severity breakdown:
  critical: 5
  high: 8
  low: 4

Sample anomalies:
  idx=102  value=6.2341  score=0.921  severity=critical  detector=ensemble
  idx=305  value=5.8773  score=0.887  severity=high      detector=ensemble
  idx=498  value=-4.9912 score=0.856  severity=high      detector=ensemble
  idx=701  value=7.1204  score=0.943  severity=critical  detector=ensemble
  idx=899  value=5.4231  score=0.812  severity=high      detector=ensemble


In [9]:
fig, ax = plt.subplots()
ax.plot(values, linewidth=0.5, alpha=0.5, color='gray', label='signal')

# color by severity
colors = {'critical': 'red', 'high': 'orange', 'warning': 'gold', 'low': 'blue'}
for sev, color in colors.items():
    sev_anomalies = [a for a in anomalies if a.severity == sev]
    if sev_anomalies:
        indices = [a.index for a in sev_anomalies]
        ax.scatter(indices, values[indices], c=color, s=30, label=sev, zorder=5)

# mark true anomalies we missed
detected_idx = {a.index for a in anomalies}
missed = [i for i in range(len(labels)) if labels[i] == 1 and i not in detected_idx]
if missed:
    ax.scatter(missed, values[missed], c='none', edgecolors='red', 
               s=50, linewidths=1, label='missed', zorder=4)

ax.set_xlabel('Time step')
ax.set_ylabel('Value')
ax.set_title('Pipeline results with severity levels')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

<Figure size 1400x400 with 1 Axes>